In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys
import copy
import math
import random
import time
import warnings
import json
import hashlib
import inspect
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, roc_auc_score
from sklearn.preprocessing import label_binarize
from imblearn.over_sampling import RandomOverSampler

# Suppress warning outputs to maintain clean logs
warnings.filterwarnings('ignore')

# =====================================================================
# GLOBAL PARAMETERS AND SYSTEM CONFIGURATION
# =====================================================================
SEED = 42
DATA_PATH = "/content/drive/MyDrive/IoT_Intrusion.csv"
TARGET_COLUMN = "label"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/pate_experiment_results.csv"

BATCH_SIZE = 128
TEACHER_EPOCHS = 5  # Operational parameter set to 5 for rapid validation across all configs
STUDENT_EPOCHS = 5
LEARNING_RATE = 1e-4
PATIENCE = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEACHER_VALID_FRACTION = 0.30
STUDENT_VALID_FRACTION = 0.20
TEACHER_POOL_FRACTION = 0.80
STUDENT_PRIVATE_FRACTION = 0.00
PUBLIC_FRACTION = 0.05
FINAL_TEST_FRACTION = 0.15

USE_OVERSAMPLING = True
LAPLACE_INVERSE_SCALE = 1.3
DELTA = 1e-5
MAX_MOMENT = 32

# =====================================================================
# DATA UTILITIES AND PREPROCESSING
# =====================================================================
def load_dataset(path, target_column):
    data = pd.read_csv(path, index_col=0)
    if target_column not in data.columns:
        raise ValueError(f"Target column '{target_column}' was not found.")
    data = data.dropna(subset=[target_column]).copy()
    if len(data) == 0:
        raise ValueError("No observations remain in the dataset.")
    return data

def preprocess_and_split(data, target_column, seed=42):
    data = data.copy()
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(data[target_column])
    X = data.drop(columns=[target_column])

    non_numeric = X.select_dtypes(include=["object", "category"]).columns.tolist()
    if len(non_numeric) > 0:
        X = X.drop(columns=non_numeric)

    X = X.apply(pd.to_numeric, errors="coerce")
    valid_rows = ~X.isna().any(axis=1)
    X = X.loc[valid_rows].reset_index(drop=True)
    y = y[valid_rows.to_numpy()]

    X = X.to_numpy(dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)

    try:
        X_teacher, X_heldout, y_teacher, y_heldout = train_test_split(
            X, y, test_size=(1.0 - TEACHER_POOL_FRACTION), random_state=seed, stratify=y
        )
    except ValueError:
        X_teacher, X_heldout, y_teacher, y_heldout = train_test_split(
            X, y, test_size=(1.0 - TEACHER_POOL_FRACTION), random_state=seed
        )

    heldout_total_sub_fraction = STUDENT_PRIVATE_FRACTION + PUBLIC_FRACTION + FINAL_TEST_FRACTION
    test_size_final_test = FINAL_TEST_FRACTION / heldout_total_sub_fraction

    try:
        X_public, X_final_test, y_public, y_final_test = train_test_split(
            X_heldout, y_heldout, test_size=test_size_final_test, random_state=seed, stratify=y_heldout
        )
    except ValueError:
        X_public, X_final_test, y_public, y_final_test = train_test_split(
            X_heldout, y_heldout, test_size=test_size_final_test, random_state=seed
        )

    X_private = np.zeros((0, X_teacher.shape[1]), dtype=np.float32)
    y_private = np.zeros((0,), dtype=np.int64)

    return X_teacher, y_teacher, X_private, y_private, X_public, y_public, X_final_test, y_final_test, label_encoder

class TabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

# =====================================================================
# PYTORCH MODEL ARCHITECTURES
# =====================================================================
class TeacherLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, num_classes=34):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:            x = x.unsqueeze(1)
        _, (hidden, _) = self.lstm(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class TeacherGRU(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, num_classes=34):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:            x = x.unsqueeze(1)
        _, hidden = self.gru(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class TeacherCNN(nn.Module):
    def __init__(self, input_size, num_classes=34):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(16 * input_size, 64)
        self.fc2 = nn.Linear(64, num_classes)
    def forward(self, x):
        if x.dim() == 2:            x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)

class TeacherMLP(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, num_classes=34):
        super().__init__()
        layers = []
        current_size = input_size
        for _ in range(num_layers):
            layers.append(nn.Linear(current_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.20))
            current_size = hidden_size
        self.network = nn.Sequential(*layers)
        self.output_layer = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 3:            x = x.mean(dim=1)
        return F.log_softmax(self.output_layer(self.network(x)), dim=1)

def build_teacher(architecture, input_size, num_classes):
    if architecture == "LSTM":
         return TeacherLSTM(input_size=input_size, num_classes=num_classes)
    if architecture == "GRU":
         return TeacherGRU(input_size=input_size, num_classes=num_classes)
    if architecture == "CNN":
        return TeacherCNN(input_size=input_size, num_classes=num_classes)
    if architecture == "MLP":
        return TeacherMLP(input_size=input_size, num_classes=num_classes)
    raise ValueError(f"Unknown architecture: {architecture}")

def build_student(architecture, input_size, num_classes):
    return build_teacher(architecture, input_size, num_classes)

# =====================================================================
# ENGINE: TRAINING AND EVALUATION
# =====================================================================
def train_model(model, train_loader, valid_loader, epochs, learning_rate, patience, device):
    model = model.to(device)
    criterion = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    best_validation_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()
        for is_input, labels in train_loader:
            is_input, labels = is_input.to(device), labels.to(device)
            optimizer.zero_grad()
            log_probabilities = model(is_input)
            loss = criterion(log_probabilities, labels)
            loss.backward()
            optimizer.step()

        model.eval()
        total_valid_loss, total_valid = 0.0, 0
        with torch.no_grad():
            for is_input, labels in valid_loader:
                is_input, labels = is_input.to(device), labels.to(device)
                log_probabilities = model(is_input)
                loss = criterion(log_probabilities, labels)
                b_size = labels.size(0)
                total_valid_loss += loss.item() * b_size
                total_valid += b_size

        valid_loss = total_valid_loss / max(total_valid, 1)
        if valid_loss < best_validation_loss:
            best_validation_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model

def predict_labels(model, loader, device):
    model.eval()
    predictions = []
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            log_probabilities = model(inputs)
            batch_predictions = torch.argmax(log_probabilities, dim=1)
            predictions.extend(batch_predictions.cpu().numpy())
    return np.asarray(predictions, dtype=np.int64)

def evaluate_student_rigorous(student_model, X_test, y_test, num_classes, batch_size, device):
    student_model.eval()
    loader = DataLoader(TabularDataset(X_test, y_test), batch_size=batch_size, shuffle=False)
    all_predictions = []
    all_probs = []

    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            log_probabilities = student_model(inputs)
            probs = torch.exp(log_probabilities)
            batch_predictions = torch.argmax(log_probabilities, dim=1)
            all_predictions.extend(batch_predictions.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_predictions = np.array(all_predictions)
    all_probs = np.array(all_probs)

    accuracy = accuracy_score(y_test, all_predictions)
    precision = precision_score(y_test, all_predictions, average='macro', zero_division=0)
    recall = recall_score(y_test, all_predictions, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, all_predictions, average='macro', zero_division=0)

    y_test_bin = label_binarize(y_test, classes=np.arange(num_classes))
    try:
        if num_classes == 2:
            auc = roc_auc_score(y_test, all_probs[:, 1])
        else:
            auc = roc_auc_score(y_test_bin, all_probs, average='macro', multi_class='ovr')
    except Exception:
        auc = np.nan

    try:
        mae = mean_absolute_error(y_test_bin, all_probs)
    except Exception:
        mae = np.nan

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_macro,
        "auc": auc,
        "mae": mae
    }

# =====================================================================
# PRIVACY ACCOUNTING & ANALYSIS
# =====================================================================
def teacher_predictions_to_counts(teacher_preds, num_classes):
    teacher_preds = np.asarray(teacher_preds, dtype=np.int64)
    num_teachers, num_queries = teacher_preds.shape
    counts = np.zeros((num_queries, num_classes), dtype=np.int64)
    for query_index in range(num_queries):
        counts[query_index] = np.bincount(teacher_preds[:, query_index], minlength=num_classes)
    return counts

def aggregate_teacher_votes_laplace(teacher_preds, num_classes, laplace_inverse_scale, random_seed=None):
    counts = teacher_predictions_to_counts(teacher_preds, num_classes)
    rng = np.random.default_rng(random_seed)
    laplace_scale = 1.0 / laplace_inverse_scale
    noise = rng.laplace(loc=0.0, scale=laplace_scale, size=counts.shape)
    noisy_counts = counts.astype(np.float64) + noise
    released_labels = np.argmax(noisy_counts, axis=1)
    return released_labels, counts, noisy_counts

def logaddexp_scalar(a, b):
    if a == -math.inf: return b
    if b == -math.inf: return a
    maximum = max(a, b)
    return maximum + math.log(math.exp(a - maximum) + math.exp(b - maximum))

def logmgf_exact(q, mechanism_eps, moment_order):
    lam = float(moment_order)
    eps = float(mechanism_eps)
    bound_1 = 0.5 * (eps ** 2) * lam * (lam + 1.0)
    bound_3 = eps * lam
    bound_2 = math.inf
    denominator = 1.0 - math.exp(eps) * q
    if q < 0.5 and denominator > 0.0:
        log_one_minus_q = math.log1p(-q)
        log_term_1 = log_one_minus_q + lam * (log_one_minus_q - math.log(denominator))
        log_term_2 = math.log(q) + eps * lam if q > 0.0 else -math.inf
        bound_2 = logaddexp_scalar(log_term_1, log_term_2)
    return min(bound_1, bound_2, bound_3)

def compute_q_noisy_max(counts, laplace_inverse_scale):
    counts = np.asarray(counts, dtype=np.float64)
    num_classes = len(counts)
    if num_classes < 2:
        return 0.0
    winner = int(np.argmax(counts))
    winning_votes = counts[winner]
    competitors = np.delete(counts, winner)
    gaps = laplace_inverse_scale * (winning_votes - competitors)
    q = float(np.sum((gaps + 2.0) * np.exp(-gaps) / 4.0))
    return min(max(q, 0.0), 1.0 - 1.0 / num_classes)

def data_dependent_log_moment(counts, laplace_inverse_scale, moment_order):
    q = compute_q_noisy_max(counts, laplace_inverse_scale)
    mechanism_eps = 2.0 * laplace_inverse_scale
    return logmgf_exact(q, mechanism_eps, moment_order)

def data_independent_log_moment(laplace_inverse_scale, moment_order):
    mechanism_eps = 2.0 * laplace_inverse_scale
    lam = float(moment_order)
    concentration_bound = 0.5 * (mechanism_eps ** 2) * lam * (lam + 1.0)
    direct_dp_bound = mechanism_eps * lam
    return min(concentration_bound, direct_dp_bound)

def calculate_pate_privacy(vote_counts, laplace_inverse_scale=1.3, delta=1e-5, moments=32):
    vote_counts = np.asarray(vote_counts, dtype=np.int64)
    num_queries, classes_dim = vote_counts.shape
    orders = np.arange(1, moments + 1, dtype=np.float64)
    dependent_total = np.zeros(moments, dtype=np.float64)
    independent_total = np.zeros(moments, dtype=np.float64)
    q_values, vote_margins = [], []

    for counts in vote_counts:
        q = compute_q_noisy_max(counts, laplace_inverse_scale)
        q_values.append(q)
        sorted_counts = np.sort(counts)
        vote_margins.append(float(sorted_counts[-1] - sorted_counts[-2]))
        for pos, order in enumerate(orders):
            dependent_total[pos] += data_dependent_log_moment(counts, laplace_inverse_scale, order)
            independent_total[pos] += data_independent_log_moment(laplace_inverse_scale, order)

    dep_candidates = (dependent_total - math.log(delta)) / orders
    ind_candidates = (independent_total - math.log(delta)) / orders
    dep_idx = np.argmin(dep_candidates)
    ind_idx = np.argmin(ind_candidates)

    dep_epsilon = float(dep_candidates[dep_idx])
    ind_epsilon = float(ind_candidates[ind_idx])

    if dep_epsilon > ind_epsilon + 1e-10:
        raise AssertionError(
            f"Invalid privacy result: dependent epsilon "
            f"{dep_epsilon} exceeds independent epsilon {ind_epsilon}"
        )

    mean_vote_margin = float(np.mean(vote_margins))

    return {
        "data_dependent_epsilon": dep_epsilon,
        "data_independent_epsilon": ind_epsilon,
        "q_values": np.asarray(q_values),
        "vote_margins": np.asarray(vote_margins),
        "mean_vote_margin": mean_vote_margin
    }

# =====================================================================
# COMPLIANCE UNIT TESTS
# =====================================================================
def test_dd_never_exceeds_di():
    test_votes = np.array([
        [9, 0, 0],
        [6, 2, 1],
        [4, 3, 2],
        [3, 3, 3]
    ])

    result = calculate_pate_privacy(
        test_votes,
        laplace_inverse_scale=1.3,
        delta=1e-5,
        moments=32
    )

    assert (
        result["data_dependent_epsilon"]
        <= result["data_independent_epsilon"] + 1e-10
    ), result
    print("PASS: test_dd_never_exceeds_di unit test verified.")

# =====================================================================
# COMPARATIVE SCENARIOS LISTING (15 SPECIFIC EXPERIMENTAL SETUPS)
# =====================================================================
configurations = [
    {"name": "Config 1", "num_teachers": 1, "teacher_archs": ["LSTM"], "student_arch": "GRU"},
    {"name": "Config 2", "num_teachers": 9, "teacher_archs": ["LSTM"] * 3 + ["CNN"] * 3 + ["MLP"] * 3, "student_arch": "LSTM"},
    {"name": "Config 3", "num_teachers": 9, "teacher_archs": ["LSTM"] * 3 + ["MLP"] * 3 + ["CNN"] * 3, "student_arch": "GRU"},
    {"name": "Config 4", "num_teachers": 3, "teacher_archs": ["LSTM"] * 3, "student_arch": "LSTM"},
    {"name": "Config 5", "num_teachers": 3, "teacher_archs": ["LSTM", "GRU", "CNN"], "student_arch": "MLP"},
    {"name": "Config 6", "num_teachers": 3, "teacher_archs": ["MLP"] * 3, "student_arch": "MLP"},
    {"name": "Config 7", "num_teachers": 3, "teacher_archs": ["CNN"] * 3, "student_arch": "CNN"},
    {"name": "Config 8", "num_teachers": 4, "teacher_archs": ["MLP", "CNN", "GRU", "LSTM"], "student_arch": "CNN"},
    {"name": "Config 9", "num_teachers": 1, "teacher_archs": ["LSTM"], "student_arch": "LSTM"},
    {"name": "Config 10", "num_teachers": 6, "teacher_archs": ["LSTM"] * 6, "student_arch": "LSTM"},
    {"name": "Config 11", "num_teachers": 12, "teacher_archs": ["CNN"] * 6 + ["GRU"] * 6, "student_arch": "CNN"},
    {"name": "Config 12", "num_teachers": 6, "teacher_archs": ["LSTM"] * 6, "student_arch": "CNN"},
    {"name": "Config 13", "num_teachers": 9, "teacher_archs": ["LSTM"] * 3 + ["GRU"] * 3 + ["CNN"] * 3, "student_arch": "MLP"},
    {"name": "Config 14", "num_teachers": 10, "teacher_archs": ["LSTM"] * 3 + ["MLP"] * 4 + ["CNN"] * 3, "student_arch": "GRU"},
    {"name": "Config 15", "num_teachers": 10, "teacher_archs": ["CNN"] * 10, "student_arch": "CNN"}
]

# =====================================================================
# CONSOLIDATED MAIN WORKFLOW
# =====================================================================
def main():
    print("Running compliance unit tests...")
    test_dd_never_exceeds_di()

    print("Loading dataset...")
    data = load_dataset(DATA_PATH, TARGET_COLUMN)
    print("Preprocessing dataset and splitting matrices...")
    X_teacher, y_teacher, _, _, X_public, y_public, X_final_test, y_final_test, label_encoder = preprocess_and_split(data, TARGET_COLUMN, seed=SEED)
    num_classes = len(label_encoder.classes_)
    input_size = X_teacher.shape[1]

    scaler = StandardScaler()
    X_teacher_scaled = scaler.fit_transform(X_teacher)
    X_public_scaled = scaler.transform(X_public)
    X_final_test_scaled = scaler.transform(X_final_test)

    results_table = []

    for idx, config in enumerate(configurations, 1):
        t0 = time.time()
        print(f"Executing Config {idx}/15: {config['name']} ({config['num_teachers']} Teachers -> {config['student_arch']} Student)")

        indices = np.arange(len(X_teacher_scaled))
        rng = np.random.default_rng(SEED)
        rng.shuffle(indices)
        partitions = np.array_split(indices, config["num_teachers"])

        teachers = []
        for t_idx, t_indices in enumerate(partitions):
            X_part = X_teacher_scaled[t_indices]
            y_part = y_teacher[t_indices]
            try:
                X_train, X_valid, y_train, y_valid = train_test_split(X_part, y_part, test_size=TEACHER_VALID_FRACTION, random_state=SEED + t_idx, stratify=y_part)
            except ValueError:
                X_train, X_valid, y_train, y_valid = train_test_split(X_part, y_part, test_size=TEACHER_VALID_FRACTION, random_state=SEED + t_idx)

            if USE_OVERSAMPLING and len(np.unique(y_train)) > 1:
                try:
                    sampler = RandomOverSampler(random_state=SEED + t_idx)
                    X_train, y_train = sampler.fit_resample(X_train, y_train)
                except Exception:
                    pass

            train_loader = DataLoader(TabularDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
            valid_loader = DataLoader(TabularDataset(X_valid, y_valid), batch_size=BATCH_SIZE, shuffle=False)

            arch = config["teacher_archs"][t_idx % len(config["teacher_archs"])]
            model = build_teacher(arch, input_size, num_classes)
            trained_model = train_model(model, train_loader, valid_loader, TEACHER_EPOCHS, LEARNING_RATE, PATIENCE, DEVICE)
            teachers.append(trained_model)

        public_loader = DataLoader(TabularDataset(X_public_scaled, y_public), batch_size=BATCH_SIZE, shuffle=False)
        all_teacher_preds = []
        for t_model in teachers:
            preds = predict_labels(t_model, public_loader, DEVICE)
            all_teacher_preds.append(preds)
        all_teacher_preds = np.stack(all_teacher_preds)

        released_labels, vote_counts, _ = aggregate_teacher_votes_laplace(all_teacher_preds, num_classes, LAPLACE_INVERSE_SCALE, random_seed=SEED)
        privacy_results = calculate_pate_privacy(vote_counts, LAPLACE_INVERSE_SCALE, DELTA, MAX_MOMENT)

        try:
            X_student_train, X_student_valid, y_student_train, y_student_valid = train_test_split(X_public_scaled, released_labels, test_size=STUDENT_VALID_FRACTION, random_state=SEED, stratify=released_labels)
        except ValueError:
            X_student_train, X_student_valid, y_student_train, y_student_valid = train_test_split(X_public_scaled, released_labels, test_size=STUDENT_VALID_FRACTION, random_state=SEED)

        student_train_loader = DataLoader(TabularDataset(X_student_train, y_student_train), batch_size=BATCH_SIZE, shuffle=True)
        student_valid_loader = DataLoader(TabularDataset(X_student_valid, y_student_valid), batch_size=BATCH_SIZE, shuffle=False)

        student_model = build_student(config["student_arch"], input_size, num_classes)
        trained_student = train_model(student_model, student_train_loader, student_valid_loader, STUDENT_EPOCHS, LEARNING_RATE, PATIENCE, DEVICE)

        metrics = evaluate_student_rigorous(trained_student, X_final_test_scaled, y_final_test, num_classes, BATCH_SIZE, DEVICE)
        elapsed = time.time() - t0

        results_table.append({
            "Configuration": config["name"],
            "dep_eps": privacy_results["data_dependent_epsilon"],
            "ind_eps": privacy_results["data_independent_epsilon"],
            "mean_vote_margin": privacy_results["mean_vote_margin"],
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1_score": metrics["f1_score"],
            "auc": metrics["auc"],
            "mae": metrics["mae"],
            "runtime_sec": elapsed
        })

    df_results = pd.DataFrame(results_table)
    print("\n=================== CONSOLIDATED EXPERIMENT COMPARATIVE RESULTS ===================")
    print(df_results.to_markdown(index=False))

    # Export directly to CSV to prevent manual copying errors
    df_results.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"\nSuccess: All results rigorously saved to {OUTPUT_CSV_PATH}")

if __name__ == "__main__":
    main()

Running compliance unit tests...
PASS: test_dd_never_exceeds_di unit test verified.
Loading dataset...
Preprocessing dataset and splitting matrices...
Executing Config 1/15: Config 1 (1 Teachers -> GRU Student)
Executing Config 2/15: Config 2 (9 Teachers -> LSTM Student)
Executing Config 3/15: Config 3 (9 Teachers -> GRU Student)
Executing Config 4/15: Config 4 (3 Teachers -> LSTM Student)
Executing Config 5/15: Config 5 (3 Teachers -> MLP Student)
Executing Config 6/15: Config 6 (3 Teachers -> MLP Student)
Executing Config 7/15: Config 7 (3 Teachers -> CNN Student)
Executing Config 8/15: Config 8 (4 Teachers -> CNN Student)
Executing Config 9/15: Config 9 (1 Teachers -> LSTM Student)
Executing Config 10/15: Config 10 (6 Teachers -> LSTM Student)
Executing Config 11/15: Config 11 (12 Teachers -> CNN Student)
Executing Config 12/15: Config 12 (6 Teachers -> CNN Student)
Executing Config 13/15: Config 13 (9 Teachers -> MLP Student)
Executing Config 14/15: Config 14 (10 Teachers -> GRU St